# One CNN Backbone, Two Computer Vision Tasks

## WORKED SOLUTION

### Part 0 — What Are We Building?

We will train two small models from scratch on cat and dog photographs:

1. **Classification:** predict one label for an image—cat or dog.
2. **Binary semantic segmentation:** predict whether every pixel is animal or background.

A **backbone** turns image pixels into **feature maps**, which are grids of learned visual features. A task-specific **head** turns those features into the required prediction.

```text
Input image [B, 3, 96, 96]
             │
             ▼
SharedBackbone architecture
             │
             ▼
Feature maps [B, 64, 24, 24]
             │
      ┌──────┴────────────────┐
      ▼                       ▼
Global pooling          1×1 convolution
Linear layer            Upsampling
      ▼                       ▼
Cat/dog logits          Animal/background mask
```

We create **two separate models**. They reuse the `SharedBackbone` design, but each has its own randomly initialised weights and trains independently.

### Learning objectives

By the end, you should be able to:

- Explain how a CNN backbone extracts image features.
- Trace tensor shapes through the backbone and two heads.
- Compare one prediction per image with one prediction per pixel.
- Explain why each task needs its own head, targets, loss and metric.

### Workshop route

| Part | Activity |
| --- | --- |
| 0 | Understand the project and set up |
| 1 | Inspect images, labels and masks |
| 2 | Build the shared backbone |
| 3 | Train the classifier |
| 4 | Train the segmenter |
| 5 | Compare both tasks on the same image |
| 6 | Review the key idea |

Keep this notebook, `requirements.txt`, and the `data` folder together. The included `data/pet_mini.npz` is about 10 MB. In Colab, upload the repository folder or upload the notebook together with `data/pet_mini.npz`.

### 0.1 Setup

**What are we doing?** Importing the libraries, setting random seeds and selecting the fastest available device.

**Why are we doing it?** PyTorch performs the tensor operations and model training. Fixed seeds make the data selection and starting weights more reproducible.

**Code:** Run the installation cell only if an import fails. Keep `requirements.txt` in the notebook's working directory.

In [ ]:
# Install into the currently selected notebook kernel if needed.
# %pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(7)
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
SIZE, CLASSES = 96, ["cat", "dog"]
LIVE_MODE = True               # 3 epochs for a workshop; False uses 10.
EPOCHS = 3 if LIVE_MODE else 10
print("Device:", device, "| Epochs:", EPOCHS)
print("Setup successful!")

**What should we observe?** The cell prints the selected device and epoch count. An **epoch** is one complete pass through the training examples. Live mode is faster but may produce weaker predictions.

## Part 1 — Understanding Our Dataset

We use the [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/), which contains pet photographs and human-created segmentation annotations.

**What are we doing?** Preparing the same photographs with two kinds of targets:

- An image-level label: `0 = cat`, `1 = dog`.
- A pixel-level mask: `0 = background`, `1 = animal`.

**Why are we doing it?** The target tells a model what question to learn. One number describes the whole image, while a mask supplies one answer at every pixel.

The original annotation also marks an animal boundary. We combine the animal and boundary into one foreground class. Images use bilinear resizing; masks use nearest-neighbour resizing so label values stay discrete.

We use a balanced prepared subset: 320 training images, 64 validation images and 64 held-out test images. Validation data monitors learning. Test data is used only after training.

**Dataset credit:** O. M. Parkhi, A. Vedaldi, A. Zisserman and C. V. Jawahar, *Cats and Dogs*, CVPR 2012. The dataset is distributed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/); image copyrights remain with their original owners.

### 1.1 Prepare the data

**Code:** The compact file already contains 96 × 96 images and masks; this cell converts them into PyTorch tensors.

In [ ]:
data = np.load("data/pet_mini.npz")


def load_split(name):
    images = torch.from_numpy(data[f"{name}_images"]).permute(0, 3, 1, 2).float() / 255
    labels = torch.from_numpy(data[f"{name}_labels"]).long()
    masks = torch.from_numpy(data[f"{name}_masks"]).unsqueeze(1).float()
    return images, labels, masks


X_train, y_train, masks_train = load_split("train")
X_valid, y_valid, masks_valid = load_split("valid")
X_test, y_test, masks_test = load_split("test")


def loader(images, targets, shuffle=False):
    return DataLoader(TensorDataset(images, targets), 32, shuffle=shuffle)


class_train = loader(X_train, y_train, True)
class_valid = loader(X_valid, y_valid)
class_test = loader(X_test, y_test)
seg_train = loader(X_train, masks_train, True)
seg_valid = loader(X_valid, masks_valid)
seg_test = loader(X_test, masks_test)

print("Train:", X_train.shape, y_train.shape, masks_train.shape)
print("Validation:", len(X_valid), "| Test:", len(X_test))
# Give foreground errors more weight because many pixels are background.
foreground_weight = (masks_train.numel() - masks_train.sum()) / masks_train.sum()

**What should we observe?**

| Tensor | Shape | Meaning |
| --- | --- | --- |
| Training images | `[320, 3, 96, 96]` | 320 RGB training images |
| Classification labels | `[320]` | One cat/dog label per training image |
| Segmentation masks | `[320, 1, 96, 96]` | One binary mask per training image |

The leading number counts examples. The image channels are red, green and blue; the mask has one channel.

### 1.2 Compare an image, its label and its mask

**What are we doing?** Displaying one dog and one cat with their labelled masks.

**Why are we doing it?** This makes the two learning targets visible before we build either model.

In [ ]:
indices = [(y_train == label).nonzero()[0, 0] for label in (1, 0)]
fig, axes = plt.subplots(2, 2, figsize=(8, 7))
for row, i in enumerate(indices):
    axes[row, 0].imshow(X_train[i].permute(1, 2, 0))
    axes[row, 0].set_title(CLASSES[y_train[i]])
    axes[row, 1].imshow(masks_train[i, 0], cmap="gray")
    axes[row, 1].set_title("Animal mask")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** White mask pixels belong to the animal and black pixels are background. The binary mask locates the animal but does not say cat or dog.

**Check your understanding:** For the dog photograph, which target answers “what is it?” and which target answers “where is it?”

## Part 2 — Building Our CNN Backbone

A **convolutional layer** slides small learned filters across an image. Its outputs are **feature maps**: spatial grids that respond to useful patterns.

- **Convolution (`Conv2d`)** learns local patterns and changes the number of channels.
- **ReLU** keeps positive responses and adds non-linearity, allowing richer patterns.
- **Pooling (`MaxPool2d`)** halves height and width while retaining strong responses.
- **Padding** adds pixels around the input so a 3 × 3 convolution can preserve spatial size.

### 2.1 See how padding changes a feature map

**What are we doing?** Applying one 3 × 3 convolution with three padding values.

**Why are we doing it?** Understanding spatial dimensions makes the larger backbone easier to follow.

For stride 1, padding 0 changes 96 × 96 to 94 × 94; padding 1 preserves 96 × 96.

In [ ]:
example = X_train[:1]
for padding in (0, 1, 2):
    output = nn.Conv2d(3, 8, 3, padding=padding)(example)
    print(f"padding={padding}: {tuple(example.shape)} -> {tuple(output.shape)}")

**What should we observe?** The outputs are `(1, 8, 94, 94)`, `(1, 8, 96, 96)` and `(1, 8, 98, 98)`. The `8` comes from `out_channels=8`.

### 2.2 Define the shared backbone

**What are we doing?** Stacking six convolution layers in three stages.

**Why are we doing it?** The backbone converts pixels into learned features. It deliberately stops before making a task-specific prediction.

**Code:** This class is the central architecture reused by both models.

In [ ]:
def conv_stage(in_channels, out_channels, pool=False):
    layers = [
        nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.ReLU(),
    ]
    return nn.Sequential(*layers, *([nn.MaxPool2d(2)] if pool else []))


class SharedBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.stage1 = conv_stage(3, 16, pool=True)
        self.stage2 = conv_stage(16, 32, pool=True)
        self.stage3 = conv_stage(32, 64)

    def forward(self, x):
        return self.stage3(self.stage2(self.stage1(x)))


backbone = SharedBackbone()
a = backbone.stage1(example)
b = backbone.stage2(a)
c = backbone.stage3(b)
print("Input:", example.shape, "→ Stage 1:", a.shape,
      "→ Stage 2:", b.shape, "→ Stage 3:", c.shape)

# Exercise 1 solution
batch_size, feature_channels, feature_height, feature_width = c.shape
print(batch_size, feature_channels, feature_height, feature_width)


**What should we observe?**

| Point | Tensor shape |
| --- | --- |
| Input | `[B, 3, 96, 96]` |
| Stage 1 | `[B, 16, 48, 48]` |
| Stage 2 | `[B, 32, 24, 24]` |
| Backbone output | `[B, 64, 24, 24]` |

Height and width shrink while feature channels increase. The backbone outputs feature maps rather than final predictions, so either task head can use them.

### Exercise 1 solution — Read the feature shape

For `example.shape == [1, 3, 96, 96]`, the batch size is `1`, the backbone produces `64` channels, and each feature map is `24 × 24`.

## Part 3 — Task A: Image Classification

**What are we doing?** Predicting one label—cat or dog—for each image.

**Why?** Classification needs one image-level answer. Global pooling removes the spatial dimensions, and a linear layer produces two **logits** (raw class scores).

```text
[B, 3, 96, 96] → backbone → [B, 64, 24, 24]
                     → global pooling → [B, 64]
                     → linear layer   → [B, 2]
```

`CrossEntropyLoss` compares the two logits with one integer class label. A **loss function** measures prediction error so training can update the weights.

### Exercise 2 solution — Classification head

The last layer needs `2` outputs: one cat logit and one dog logit.

In [ ]:
# Exercise 2 solution
NUM_CLASSES = 2


class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SharedBackbone()
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x):
        return self.head(self.backbone(x))


classifier = CNNClassifier().to(device)
print(classifier)


**What should we observe?** The printed model has a `backbone` and a classification `head`. Its final layer has two outputs.

### Exercise 3 solution — Classification loss

Use `nn.CrossEntropyLoss()` because cat and dog are mutually exclusive classes. Pass raw logits to the loss.

### 3.2 Train the classifier

The randomly initialised filters learn by repeatedly predicting, measuring loss and updating their weights.

In [ ]:
def average_loss(model, data_loader, loss_fn):
    model.eval()
    with torch.no_grad():
        losses = [loss_fn(model(x.to(device)), y.to(device)).item()
                  for x, y in data_loader]
    return sum(losses) / len(losses)


def fit(model, train_loader, valid_loader, loss_fn, epochs=EPOCHS):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f"Epoch {epoch:2}: train={sum(losses)/len(losses):.3f} | "
              f"valid={average_loss(model, valid_loader, loss_fn):.3f}")
    return model


# Exercise 3 solution
classification_loss = nn.CrossEntropyLoss()
classifier = fit(classifier, class_train, class_valid,
                 classification_loss, epochs=EPOCHS)


**What should we observe?** Training and validation loss should generally decrease, although they may fluctuate. One epoch is one full pass through the training subset. Exact results vary by device and run.

### 3.3 Evaluate and view predictions

**What are we doing?** Predicting labels for held-out photographs and calculating accuracy.

**Why are we doing it?** **Accuracy** is the fraction of test images assigned the correct cat/dog label. Held-out images give a fairer check than training images.

In [ ]:
classifier.eval()
images, labels = next(iter(class_test))
with torch.no_grad():
    logits = classifier(images.to(device)).cpu()
predictions = logits.argmax(1)

correct = 0
with torch.no_grad():
    for x, y in class_test:
        correct += (classifier(x.to(device)).argmax(1).cpu() == y).sum().item()
print(f"Test accuracy: {correct / len(class_test.dataset):.1%}")

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, image, actual, predicted in zip(axes, images, labels, predictions):
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(f"True: {CLASSES[actual]}\nPred: {CLASSES[predicted]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** Each output row contains two logits. `argmax(dim=1)` selects the larger score. The plot shows the true and predicted label for the same fixed examples, whether the predictions are correct or incorrect.

This model starts from random weights and sees a small subset, so it may perform much worse than the pretrained MobileNetV2 used earlier.

## Part 4 — Task B: Image Segmentation

Classification asks **“cat or dog?”** Segmentation asks **“which pixels belong to the animal?”**

**What changes?** We create a fresh `SharedBackbone` instance and attach a spatial head:

```text
[B, 3, 96, 96] → backbone → [B, 64, 24, 24]
                     → 1×1 convolution → [B, 1, 24, 24]
                     → upsampling      → [B, 1, 96, 96]
```

A 1×1 convolution produces one animal logit at each feature-map location. Bilinear upsampling reconstructs the required image-sized grid.

### Exercise 4 solution — Segmentation head

The 1×1 convolution needs `1` output channel because each pixel has one binary animal/background decision.

In [ ]:
# Exercise 4 solution
MASK_CHANNELS = 1


class CNNSegmenter(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = SharedBackbone()  # Fresh weights.
        self.head = nn.Sequential(
            nn.Conv2d(64, MASK_CHANNELS, 1),
            nn.Upsample((SIZE, SIZE), mode="bilinear", align_corners=False),
        )

    def forward(self, x):
        return self.head(self.backbone(x))


segmenter = CNNSegmenter().to(device)
print(segmenter)


In [ ]:
# Exercise 5 solution
segmentation_loss = nn.BCEWithLogitsLoss(
    pos_weight=foreground_weight.to(device)
)
segmenter = fit(segmenter, seg_train, seg_valid,
                segmentation_loss, epochs=EPOCHS)


**What should we observe?** The segmenter contains the same backbone architecture and a different head. Its backbone has fresh weights; it does not reuse the classifier's trained parameters.

### Exercise 5 solution — Segmentation loss

Use `nn.BCEWithLogitsLoss()` for one binary logit per pixel. It expects raw logits and applies sigmoid internally. `pos_weight` prevents the larger background area from dominating this small dataset.

### 4.2 Evaluate and view predicted masks

For display, sigmoid converts logits into scores from 0 to 1, and a threshold of 0.5 creates a binary mask.

**Dataset-level foreground IoU** is total foreground intersection divided by total foreground union across the test set. It focuses on animal overlap rather than the often-large background.

In [ ]:
segmenter.eval()
images, true_masks = next(iter(seg_test))
with torch.no_grad():
    predicted_masks = segmenter(images.to(device)).sigmoid().cpu() >= 0.5

intersection = union = 0
with torch.no_grad():
    for x, y in seg_test:
        prediction = segmenter(x.to(device)).sigmoid().cpu() >= 0.5
        truth = y.bool()
        intersection += (prediction & truth).sum().item()
        union += (prediction | truth).sum().item()
print(f"Dataset-level foreground IoU: {intersection / union:.3f}")

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for row in range(3):
    axes[row, 0].imshow(images[row].permute(1, 2, 0))
    axes[row, 1].imshow(true_masks[row, 0], cmap="gray")
    axes[row, 2].imshow(predicted_masks[row, 0], cmap="gray")
for ax, title in zip(axes[0], ["Image", "True mask", "Predicted mask"]):
    ax.set_title(title)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

**What should we observe?** The columns show the photograph, ground-truth mask and predicted mask. Boundaries may look coarse because pooling reduces the feature maps to 24 × 24 before upsampling. Bilinear upsampling enlarges predictions; it does not restore detail that was lost.

This is binary semantic segmentation: all animal pixels share one foreground class. It does not separate multiple animal instances.

## Part 5 — Comparing Both Tasks

| Component | Classification | Segmentation |
| --- | --- | --- |
| Input | Image | Image |
| Backbone | `SharedBackbone` | `SharedBackbone` |
| Output head | Global pooling + linear | 1×1 convolution + upsampling |
| Prediction | One class per image | One class per pixel |
| Target | Cat/dog label | Animal/background mask |
| Loss | `CrossEntropyLoss` | `BCEWithLogitsLoss` |
| Metric | Accuracy | Dataset-level foreground IoU |

The architecture stays the same through the backbone. The head, targets, loss and metric change. The models use separate backbone instances and train independently from scratch.

### Same test image, two different outputs

In [ ]:
image = X_test[:1]
true_label, true_mask = y_test[0].item(), masks_test[0, 0]
classifier.eval()
segmenter.eval()
with torch.no_grad():
    class_prediction = classifier(image.to(device)).argmax(1).item()
    mask_prediction = segmenter(image.to(device)).sigmoid()[0, 0].cpu() >= 0.5

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(image[0].permute(1, 2, 0))
axes[0].set_title("Original image")
axes[1].imshow(image[0].permute(1, 2, 0))
axes[1].set_title(f"Class: {CLASSES[class_prediction]}\nTrue: {CLASSES[true_label]}")
axes[2].imshow(true_mask, cmap="gray")
axes[2].set_title("Ground-truth mask")
axes[3].imshow(mask_prediction, cmap="gray")
axes[3].set_title("Predicted mask")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 6 — Key Takeaways

1. A CNN backbone extracts feature maps from image pixels.
2. Both tasks can reuse the same backbone architecture.
3. Classification collapses space to predict one class per image.
4. Segmentation reconstructs space to predict one label per pixel.
5. The task-specific head, targets and loss determine what the model learns.
6. Our two models have separate weights and train independently from scratch.

### Exercise 6 solution — Compare the models

- The input format, backbone architecture and general training steps stay the same.
- Classification outputs `[B, 2]`; segmentation outputs `[B, 1, H, W]` because it must retain a prediction for every pixel.
- The losses differ because the targets have different meanings and shapes.
- A trained classification head cannot directly perform segmentation because it discards spatial information and returns image-level scores.